In [1]:
from _setup import setup_project_root
PROJECT_ROOT = setup_project_root()

In [2]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from models.lstm import build_lstm_sequences, get_lstm_model
import numpy as np
from evaluation.metrics import compute_metrics, compute_metrics_real_scale
import joblib
import tensorflow as tf

In [3]:
train_df = pd.read_csv(PROJECT_ROOT / "data" / "station_split" / "train.csv")
val_df   = pd.read_csv(PROJECT_ROOT / "data" / "station_split" / "val.csv")
test_df  = pd.read_csv(PROJECT_ROOT / "data" / "station_split" / "test.csv")

In [4]:
pm25_scaler = joblib.load(PROJECT_ROOT / "artifacts" / "pm25_scaler.pkl")

c:\Users\KimNgan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
train_df['PM2.5_scaled'] = pm25_scaler.fit_transform(train_df[['PM2.5']])
val_df['PM2.5_scaled']   = pm25_scaler.transform(val_df[['PM2.5']])
test_df['PM2.5_scaled']  = pm25_scaler.transform(test_df[['PM2.5']])

In [6]:
X_train, y_train, _ = build_lstm_sequences(train_df, 'PM2.5_scaled')
X_val, y_val, _     = build_lstm_sequences(val_df, 'PM2.5_scaled')
X_test, y_test, _   = build_lstm_sequences(test_df, 'PM2.5_scaled')

model = get_lstm_model(input_shape=(24, 1))
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train, 
    validation_data=(X_val, y_val),
    epochs=50, batch_size=64, callbacks=[early_stop]
)

Epoch 1/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - loss: 0.0254 - val_loss: 0.0210
Epoch 2/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - loss: 0.0205 - val_loss: 0.0203
Epoch 3/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - loss: 0.0201 - val_loss: 0.0205
Epoch 4/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - loss: 0.0200 - val_loss: 0.0199
Epoch 5/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 24s 27ms/step - loss: 0.0199 - val_loss: 0.0203
Epoch 6/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 39s 25ms/step - loss: 0.0198 - val_loss: 0.0200
Epoch 7/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 24s 27ms/step - loss: 0.0198 - val_loss: 0.0199
Epoch 8/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 24s 27ms/step - loss: 0.0197 - val_loss: 0.0200
Epoch 9/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 39s 24ms/step - loss: 0.0197 - val_loss: 0.0199
Epoch 10/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 40s 23ms/step - loss: 0.0196 - val_loss: 0.0202
Epoch 11/50
865/865 ━━━━━━━━━━━━━━━━━━━━ 21s 25ms/step - loss: 0.0195 - val_loss: 0.0200
Epoch 12/50
865/865 ━━━━━━━━━━

In [7]:
pred_scaled = model.predict(X_test)
pred_real = pm25_scaler.inverse_transform(pred_scaled)
y_real = pm25_scaler.inverse_transform(y_test)

res = compute_metrics(y_real, pred_real)
print("Station LSTM Metrics:", res)

209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Station LSTM Metrics: {'Correlation': 0.5511180214419504, 'RMSE': 0.14340399721754896, 'MAPE': 2239.0410227767616, 'MAE': 0.09915172074173245}


In [8]:
metrics= compute_metrics_real_scale(
    y_test,      
    pred_scaled,   
    scaler=pm25_scaler
)
metrics

{'Correlation': 0.5511180214419504,
 'RMSE': 0.14340399721754896,
 'MAPE': 0.4998029448367931,
 'MAE': 0.09915172074173245}

In [9]:
metrics= compute_metrics_real_scale(
    y_test[:, 0],      
    pred_scaled[:, 0],   
    scaler=pm25_scaler
)
metrics

{'Correlation': 0.9000330440840308,
 'RMSE': 0.07459152556026016,
 'MAPE': 0.19249801635742186,
 'MAE': 0.04743586524897329}

In [10]:
import numpy as np
import pandas as pd

if 'pred_scaled' not in locals() or 'y_test' not in locals():
    print("❌ Bạn cần chạy bước dự báo (model.predict) trước!")
else:
    station_ids = test_df["Station_No"].values[-len(y_test):]
    rows = []
    for sid in np.unique(station_ids):
        mask = (station_ids == sid)
        
        if mask.sum() < 50: 
            continue
            
        y_true_sub = y_test[mask, 0] if y_test.ndim > 1 else y_test[mask]
        y_pred_sub = pred_scaled[mask, 0] if pred_scaled.ndim > 1 else pred_scaled[mask]
        
        try:
            met = compute_metrics_real_scale(
                y_true_sub, 
                y_pred_sub, 
                scaler=pm25_scaler, 
                mape_threshold=1.0
            )
            rows.append({"Station_No": sid, "n": int(mask.sum()), **met})
        except Exception as e:
            continue

    if rows:
        df_station_metrics = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
        display(df_station_metrics.head(10)) 
    else:
        print("❌ Không tìm thấy dữ liệu phù hợp.")

,Station_No,n,Correlation,RMSE,MAPE,MAE
0,4,1112,0.915834,0.068642,0.174985,0.045870
1,1,1113,0.910626,0.070272,0.228405,0.045525
2,5,1112,0.897436,0.075447,0.211618,0.046236
3,2,1113,0.890849,0.076841,0.125664,0.047861
4,3,1113,0.896658,0.077106,0.188536,0.049007
5,6,1112,0.889522,0.078685,0.217177,0.050116
